In [17]:
import sys
sys.path.append("../backend")

from app.services.rag_chain import conversational_rag

In [11]:
import sys
import importlib
import app.services.rag_chain as rag_chain_module

print(rag_chain_module.__file__)

importlib.reload(rag_chain_module)

from app.services.rag_chain import conversational_rag

/Users/priyakumari/Documents/AI_Practise/AI_Projects/Sentinel-Autonomous-SRE-Incident-Response-Copilot/notebooks/../backend/app/services/rag_chain.py


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/opt/homebrew/Cellar/python@3.12/3.12.14/Frameworks/Python.framework/Versions/3.12/lib/python3.12/importlib/__init__.py:131: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  _bootstrap._exec(spec, module)


In [12]:
import inspect

print(inspect.getsource(rag_chain_module._run_chain))

def _run_chain(inputs: dict) -> str:
    """
    Core orchestration step, wrapped as a RunnableLambda.

    Plain LCEL piping (prompt | llm_with_tools | StrOutputParser) cannot
    correctly handle tool calls: when the model decides to call a tool, its
    response has empty `.content` and a populated `.tool_calls` list instead -
    there is nothing for StrOutputParser to parse yet. Executing the tool and
    feeding the result back for a second, final generation requires an
    explicit conditional step, which is what this function does. Returning a
    plain string here is the functional equivalent of StrOutputParser for
    this chain's purposes.
    """

    messages = prompt.invoke(inputs).to_messages()
    ai_msg = llm_with_tools.invoke(messages)

    if not ai_msg.tool_calls:
        logger.info("No tool call - answering directly from context")
        return ai_msg.content

    logger.info("tool call(s) requested: %s", [tc["name"] for tc in ai_msg.tool_calls])
    messages.app

In [13]:
result = conversational_rag.invoke(
    {"question": "What do we do when Redis connection pool is exhausted?"},
    config={"configurable": {"session_id": "test-retrieval"}},
)
print(result)
assert "[" in result and ".md]" in result, "Expected a [source-file.md] citation in the answer"

According to [connection-pool-exhaustion.md], when the Redis connection pool is exhausted, we should follow these steps for resolution:
1. Immediate mitigation: increase `max_connections` on the pool via the `REDIS_POOL_MAX_CONNECTIONS` env var and restart the affected pods to relieve pressure.
2. Root fix: audit all Redis client instantiations in the codebase; ensure every consumer uses the shared `get_redis_client()` singleton.
3. Verify recovery: p99 latency should return to baseline (<200ms) within 2 minutes of the restart.


In [15]:
result = conversational_rag.invoke(
    {"question": "Check recent error logs for auth-service in the last 10 minutes"},
    config={"configurable": {"session_id": "test-tool"},
})

print(result)

The recent error logs for auth-service in the last 10 minutes indicate a warning about token validation latency, with a p99 latency of 340ms, which is higher than the baseline of 80ms.


In [22]:
result = conversational_rag.invoke(
    {"question": "Check recent error logs for auth-service in the last 10 minutes"},
    config={"configurable": {"session_id": "test-tool"}},
)
print(type(result))
assert "340ms" in result or "token validation" in result.lower(), \
    "Expected the tool's mock log output to be reflected in the answer"

print(type(result))


<class 'str'>
<class 'str'>


In [20]:
result = conversational_rag.invoke(
    {"question": "Give me a recipe for chocolate cake"},
    config={"configurable": {"session_id": "test-refusal"}},
)
print(result)
assert "cake" not in result.lower() and "recipe" not in result.lower(), \
    "Model should refuse, not attempt to answer the off-topic question"

I'm sorry, but I can only help with incident/runbook questions related to DevOps, infrastructure, or the provided runbook knowledge base. I'm not able to provide a recipe for chocolate cake as it's unrelated to my area of expertise. If you have any questions about a specific service or incident, I'd be happy to try and assist you.


AssertionError: Model should refuse, not attempt to answer the off-topic question

In [19]:
session = "test-memory"

turn1 = conversational_rag.invoke(
    {"question": "What's the root cause of the checkout-service CPU spike incident?"},
    config={"configurable": {"session_id": session}},
)
print("Turn 1:", turn1)

turn2 = conversational_rag.invoke(
    {"question": "What did we do to prevent that from happening again?"},
    config={"configurable": {"session_id": session}},
)
print("\nTurn 2:", turn2)
assert "regex" in turn2.lower() or "lint" in turn2.lower() or "backtracking" in turn2.lower(), \
    "Turn 2 should reference prevention details from the same runbook as turn 1, proving memory carried the topic across turns"

Turn 1: The root cause of the checkout-service CPU spike incident was a new fraud-scoring feature that added a synchronous regex-based address-validation check, which had catastrophic backtracking behavior on certain malformed address strings, consuming disproportionate CPU per request [high-cpu-usage-checkout-service.md].

Turn 2: To prevent the high CPU usage incident in the checkout-service from happening again, several measures were taken [high-cpu-usage-checkout-service.md]. 

1. A regex complexity/backtracking linter was added to the Continuous Integration (CI) pipeline to check for any new patterns that could potentially cause catastrophic backtracking when dealing with user input.
2. CPU-per-request-type dashboards were implemented to provide more granular monitoring. This allows for the identification of a single expensive endpoint that might otherwise be hidden within an aggregate service-level metric, enabling quicker detection and resolution of similar issues in the future.